# URLS

In [1]:
# install.packages("here")  # solo la primera vez
library(here)

root <- here()
# dataset <- fread(file.path(path_base, "competencia_01_crudo.csv"))
list.files(root)
dataset_folder <- file.path(root, "DATA", "DATASETS")
exp_folder <- file.path(root, "DATA", "EXP")

dataset_url <- file.path(dataset_folder,"competencia_01_crudo.csv")

here() starts at /home/marco/code/DMEyF/dmeyf2026



[1] "arboles"          "CazaTalentos"     "DATA"             "ensembles"       
[5] "git-zero-to-hero" "mis_pruebas"      "monday"           "zero2hero"

# Generacion clase ternaria

In [2]:
require( "data.table" )

# leo el dataset
# dataset <- fread("/content/datasets/competencia_01_crudo.csv" )
dataset <- fread(dataset_url)

# calculo el periodo0 consecutivo
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    # file =  "/content/datasets/competencia_01.csv.gz",
    file = file.path(dataset_folder,"competencia_01.csv.gz"),
    sep = ","
)

Loading required package: data.table



In [3]:
setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente)
dataset[, .N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


# Librerias

In [4]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: parallel

Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, warnings


Loading required package: primes

Loading required package: rlist

Warning message:
“package ‘

# Parametros

In [5]:
PARAM <- list()
PARAM$experimento <- 4942
PARAM$semilla_primigenia <- 240707

In [6]:
# training y future
PARAM$train <- c(202104)
PARAM$train_final <- c(202104)
PARAM$future <- c(202106)
PARAM$semilla_kaggle <- 314159
PARAM$cortes <- seq(4000, 19000, by= 500)

In [7]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.1

In [8]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "average_precision",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  # min_data_in_leaf= 184,     # fijo, esto es FUNDAMENTAL
  min_data_in_leaf= 1840,     # fijo, esto es FUNDAMENTAL
  feature_fraction= 0.5707911,  # fijo
  learning_rate= 0.0020182,    # fijo

  num_iterations= 2505,     # se modificara
  num_leaves= 61,          # se modificara
  min_sum_hessian_in_leaf= 0.0067274  # se modificara
)


In [9]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [10]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [11]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]

  prealidad[, predicted:=NULL]
  return( res )
}

In [12]:
# lectura del dataset
# dataset <- fread("/content/datasets/competencia_01.csv.gz", stringsAsFactors= TRUE)
dataset <- fread(file.path(dataset_folder,"competencia_01.csv.gz"), stringsAsFactors= TRUE)

In [13]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [14]:
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N, list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<fct>,<int>
202104,CONTINUA,161181
202104,BAJA+2,1139
202104,BAJA+1,964


In [15]:
PARAM$lgbm$param_fijos




$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "average_precision"

$feature_pre_filter
[1] FALSE

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$force_row_wise
[1] TRUE

$deterministic
[1] TRUE

$verbosity
[1] -100

$seed
[1] 240707

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$early_stopping
[1] 0

$min_data_in_leaf
[1] 1840

$feature_fraction
[1] 0.5707911

$learning_rate
[1] 0.0020182

$num_iterations
[1] 2505

$num_leaves
[1] 61

$min_sum_hessian_in_leaf
[1] 0.0067274

In [16]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes %in% PARAM$future]

In [17]:
PARAM$future

[1] 202106

In [18]:
campos_buenos1 <- setdiff(
  colnames(dataset),
  c("clase_ternaria", "clase01", "azar", "training")
)
campos_buenos2 <- setdiff(
  colnames(dataset),
  c(
    "clase_ternaria", "clase01", "azar", "training",
    "cprestamos_personales", "mprestamos_personales"
  )
)

In [19]:
# Entreno el modelo sobre todo el periodo

dtrain_final1 <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos1, with= FALSE]),
  label= dataset_train[, clase01]
)

modelo_final1 <- lgb.train(
  data= dtrain_final1,
  param= PARAM$lgbm$param_fijos
)

In [20]:
# Entreno el modelo sobre todo el periodo

dtrain_final2 <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos2, with= FALSE]),
  label= dataset_train[, clase01]
)

modelo_final2 <- lgb.train(
  data= dtrain_final2,
  param= PARAM$lgbm$param_fijos
)

# Evaluacion 1
    removemos columnas "clase_ternaria", "clase01", "azar", "training"
    Los Hiperparametros fueron encotrados con BO usando esta configuracion de columnas de partida en el dataset
    

In [21]:
# hago el predict en los datos del futuro, y calculo la ganancia

# aplico el modelo a los datos nuevos
prediccion1 <- predict(
  modelo_final1,
  data.matrix(dfuture[, campos_buenos1, with= FALSE])
)


In [22]:
# calculo la ganancia en los datos ddel futuro para todos los cortes
tb_prediccion1 <- dfuture[, list(numero_de_cliente, foto_mes,clase_ternaria)]
tb_prediccion1[, prob := prediccion1 ]
tb_prediccion1[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]

# lgb.save(modelo_final, "modelo.txt" )
# Dibujo la curva de ganancia acumulada
setorder(tb_prediccion1, -prob)
tb_prediccion1[, ganancia_acumulada := cumsum(gan)]
tb_prediccion1[, pos := sequence(.N)]

for (envios in PARAM$cortes) {
  ganancia1 <- tb_prediccion1[1:envios, sum(gan)]
  cat( envios, "\t", ganancia1, "\n")
}

4000 	 255200000 
4500 	 283250000 
5000 	 293700000 
5500 	 315150000 
6000 	 322300000 
6500 	 331650000 
7000 	 335500000 
7500 	 344850000 
8000 	 350900000 
8500 	 352550000 
9000 	 355300000 
9500 	 359150000 
10000 	 359700000 
10500 	 364650000 
11000 	 364100000 
11500 	 363550000 
12000 	 361900000 
12500 	 371250000 
13000 	 372900000 
13500 	 381150000 
14000 	 380600000 
14500 	 377850000 
15000 	 378400000 
15500 	 380050000 
16000 	 376200000 
16500 	 372350000 
17000 	 364100000 
17500 	 356950000 
18000 	 356400000 
18500 	 353650000 
19000 	 346500000 


# Evaluacion 2
        Removemos "clase_ternaria", "clase01", "azar", "training",    "cprestamos_personales", "mprestamos_personales"
        Los Hiperparametros fueron encotrados con BO con "cprestamos_personales", "mprestamos_personales" presentes, no se habían removido.

In [23]:
# hago el predict en los datos del futuro, y calculo la ganancia

# aplico el modelo a los datos nuevos
prediccion2 <- predict(
  modelo_final2,
  data.matrix(dfuture[, campos_buenos2, with= FALSE])
)


In [25]:
# calculo la ganancia en los datos ddel futuro para todos los cortes
tb_prediccion2 <- dfuture[, list(numero_de_cliente, foto_mes,clase_ternaria)]
tb_prediccion2[, prob := prediccion2 ]
tb_prediccion2[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]

# lgb.save(modelo_final, "modelo.txt" )
# Dibujo la curva de ganancia acumulada
setorder(tb_prediccion2, -prob)
tb_prediccion2[, ganancia_acumulada := cumsum(gan)]
tb_prediccion2[, pos := sequence(.N)]


In [27]:

for (envios in PARAM$cortes) {
  ganancia2 <- tb_prediccion2[1:envios, sum(gan)]
  cat( envios, "\t", ganancia2, "\n")}

4000 	 314600000 
4500 	 332750000 
5000 	 345400000 
5500 	 358050000 
6000 	 371800000 
6500 	 389950000 
7000 	 392700000 
7500 	 396550000 
8000 	 409200000 
8500 	 418550000 
9000 	 420200000 
9500 	 430650000 
10000 	 438900000 
10500 	 447150000 
11000 	 442200000 
11500 	 443850000 
12000 	 445500000 
12500 	 451550000 
13000 	 4.51e+08 
13500 	 442750000 
14000 	 442200000 
14500 	 441650000 
15000 	 438900000 
15500 	 432850000 
16000 	 426800000 
16500 	 424050000 
17000 	 422400000 
17500 	 420750000 
18000 	 411400000 
18500 	 406450000 
19000 	 3.96e+08 
